In [1]:
!pip install -q pydantic ollama


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import ollama
import re
from typing import Literal
from pydantic import BaseModel, Field

In [ ]:
try:
    prompt_urgencia = open("prompt_urgencia.md", "r", encoding="utf-8").read()
    prompt_categoria = open("prompt_categoria.md", "r", encoding="utf-8").read()
    prompt_accion = open("prompt_accion.md", "r", encoding="utf-8").read()
    prompt_respuesta = open("prompt_respuesta.md", "r", encoding="utf-8").read()
except FileNotFoundError as e:
    raise RuntimeError(f"Error al leer los archivos de prompt: {e}")

In [ ]:
class SalidaUrgencia(BaseModel):
    nivel_urgencia: Literal["Alta", "Baja"]

class SalidaCategoria(BaseModel):
    categoria: Literal["Envío/Logística", "Facturación", "Producto Defectuoso"]

class SalidaAccion(BaseModel):
    cadena_de_pensamiento: str
    politica_aplicada: Literal["Generar ticket", "Reembolsar", "Pedir más datos", "Reprogramar envío"]
    codigo_accion_sistema: Literal["ACC_GENERAR_TICKET", "ACC_REEMBOLSO", "ACC_PEDIR_DATOS", "ACC_REPROGRAMAR_ENVIO"]

class Metadatos(BaseModel):
    categoria: str
    nivel_urgencia: str

class Analisis(BaseModel):
    cadena_de_pensamiento: str
    politica_aplicada: str

class AccionRespuesta(BaseModel):
    codigo_accion_sistema: str
    mensaje_al_cliente: str

class SalidaMaestra(BaseModel):
    metadatos_clasificacion: Metadatos
    analisis_interno: Analisis
    accion_y_respuesta: AccionRespuesta

In [ ]:
class OutputValidator:
    def __init__(self):
        self.suspicious_patterns = [
            r'##\s*ROL\s+Sos\s+un',
            r'##\s*(INSTRUCCI[OÓ]N|REGLAS DE SEGURIDAD|TONO|VALORES PERMITIDOS|EJEMPLOS|INPUT|OUTPUT)',
            r'API[_\s]KEY[:=]\s*\w+',
        ]

    def validate_output(self, output: str) -> bool:
        return not any(re.search(pattern, output, re.IGNORECASE)
                      for pattern in self.suspicious_patterns)

    def filter_response(self, response: str) -> str:
        if not self.validate_output(response) or len(response) > 1500:
            return "No puedo procesar solicitudes que entren en conflicto con mis directrices operativas."
        return response

In [ ]:
def llamar_llm(system_prompt: str, input_data: dict) -> dict:
    mensajes = [
        {
            "role": "system",
            "content": f"{system_prompt}\n\nDebes responder ÚNICAMENTE con un objeto JSON válido."
        },
        {
            "role": "user",
            "content": json.dumps(input_data)
        }
    ]
    respuesta = ollama.chat(
        model="qwen3:8b",
        messages=mensajes,
        format="json",
        options={"temperature": 0.1}
    )
    return json.loads(respuesta['message']['content'])

In [ ]:
def procesar_mensaje(mensaje_usuario: str) -> dict:

    validator = OutputValidator()

    print("Procesado urgencia...")
    out_urgencia = llamar_llm(prompt_urgencia, {"mensaje_usuario": mensaje_usuario})
    nivel_urgencia = out_urgencia.get("nivel_urgencia", "")

    print("Procesado categoria...")
    out_categoria = llamar_llm(prompt_categoria, {"mensaje_usuario": mensaje_usuario})
    categoria = out_categoria.get("categoria", "")

    print("Procesado accion...")
    out_accion = llamar_llm(
        prompt_accion,
        {"mensaje_usuario": mensaje_usuario, "nivel_urgencia": nivel_urgencia, "categoria": categoria}
    )
    cadena_de_pensamiento = out_accion.get("cadena_de_pensamiento", "")
    politica_aplicada = out_accion.get("politica_aplicada", "")
    codigo_accion_sistema = out_accion.get("codigo_accion_sistema", "")

    print("Procesado respuesta...")
    out_respuesta = llamar_llm(
        prompt_respuesta,
        {"mensaje_usuario": mensaje_usuario, "nivel_urgencia": nivel_urgencia, "categoria": categoria, "politica_aplicada": politica_aplicada}
    )
    mensaje_al_cliente = validator.filter_response(out_respuesta.get("mensaje_al_cliente", ""))

    salida_estructurada = {
        "metadatos_clasificacion": {
            "categoria": categoria,
            "nivel_urgencia": nivel_urgencia
        },
        "analisis_interno": {
            "cadena_de_pensamiento": cadena_de_pensamiento,
            "politica_aplicada": politica_aplicada
        },
        "accion_y_respuesta": {
            "codigo_accion_sistema": codigo_accion_sistema,
            "mensaje_al_cliente": mensaje_al_cliente
        }
    }

    salida_validada = SalidaMaestra(**salida_estructurada)
    return salida_validada.model_dump()